# Complete Step-by-Step LoRA (Low-Rank Adaptation) Tutorial

This notebook provides an in-depth, hands-on tutorial for understanding, implementing, training, and deploying **LoRA (Low-Rank Adaptation)** in PyTorch from scratch, followed by the industry-standard implementation using Hugging Face PEFT.

---

## 1. Mathematical Foundations of LoRA

In traditional fine-tuning, when adjusting a pre-trained weight matrix $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$, the forward pass computes:

$$h = (W_0 + \Delta W)x$$

where $\Delta W \in \mathbb{R}^{d_{out} \times d_{in}}$ represents the full-rank weight update matrix.

### The Low-Rank Hypothesis
Aghajanyan et al. and Hu et al. observed that model adaptation has a low **intrinsic rank**. Instead of learning all elements of $\Delta W$, LoRA decomposes $\Delta W$ into the product of two low-rank matrices:

$$\Delta W = \frac{\alpha}{r} (B \cdot A)$$

where:
- $A \in \mathbb{R}^{r \times d_{in}}$ (down-projection)
- $B \in \mathbb{R}^{d_{out} \times r}$ (up-projection)
- $r \ll \min(d_{in}, d_{out})$ is the rank hyperparameter (e.g., $r = 4$ or $r = 8$)
- $\alpha$ is a scaling hyperparameter (typically set to $2r$ or $16$)

The forward pass therefore becomes:

$$h = W_0 x + \frac{\alpha}{r} (B \cdot A) x$$

### Initialization Strategy
- **Matrix $A$** is initialized with a Gaussian random distribution ($A \sim \mathcal{N}(0, \sigma^2)$ or Kaiming uniform).
- **Matrix $B$** is initialized strictly to **zero** ($B = 0$).
- At step 0 of training: $\Delta W = B \cdot A = 0 \cdot A = 0$. Consequently, the model begins with the exact behavior and predictions of the base model without initial distortion.

## 2. Environment Setup

Let's import PyTorch, its neural network components, and mathematical utilities.

In [ ]:
import math
import torch
import torch.nn as nn

# Set random seeds for deterministic reproducibility
torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 3. Implementing `LoRALinear` from Scratch

Below is the complete implementation of a PyTorch module that wraps a standard `nn.Linear` layer with LoRA adapters.

### Detailed Explanation of Each Component:
1. `self.base_layer`: Holds the reference to the original `nn.Linear` layer.
2. `self.base_layer.weight.requires_grad = False`: Freezes the original base weights so no gradients or optimizer momentum buffers are allocated for them.
3. `self.lora_A`: Trainable parameter tensor of shape `(r, in_features)`.
4. `self.lora_B`: Trainable parameter tensor of shape `(out_features, r)`.
5. `self.scaling = alpha / r`: Constant factor that normalizes the adapter's update magnitude across different choices of rank $r$.
6. `nn.init.kaiming_uniform_(self.lora_A)`: Provides variance-preserving initial weights for the down-projection.
7. `nn.init.zeros_(self.lora_B)`: Ensures $\Delta W = 0$ at initialization.
8. `forward()`: Calculates the base output $W_0 x + b$ and adds the scaled adapter path $(x A^T B^T) \times \text{scaling}$.

In [ ]:
class LoRALinear(nn.Module):
    """
    Custom linear layer wrapping an existing nn.Linear with Low-Rank Adaptation (LoRA).
    """
    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        
        # Store base layer reference and configuration hyperparameters
        self.base_layer = base_layer
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        
        in_features = base_layer.in_features
        out_features = base_layer.out_features
        
        # Step 1: Freeze all parameters in the base linear layer
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
            
        # Step 2: Define low-rank trainable weight matrices A and B
        # Matrix A compresses from in_features down to rank r
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        # Matrix B expands from rank r back up to out_features
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        
        # Step 3: Initialize A with Kaiming uniform distribution and B with exact zeros
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Compute output from the original pre-trained frozen base layer: (batch, out_features)
        base_out = self.base_layer(x)
        
        # 2. Compute output through low-rank adapter path:
        # Input x: (batch_size, in_features)
        # x @ lora_A.T: (batch_size, in_features) @ (in_features, r) -> (batch_size, r)
        # (x @ lora_A.T) @ lora_B.T: (batch_size, r) @ (r, out_features) -> (batch_size, out_features)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        
        # 3. Sum the base output and scaled adapter delta
        return base_out + self.scaling * lora_out

## 4. Building and Adapting a Model

Let's construct a small neural network representing a pre-trained model and replace its linear layer with our `LoRALinear` module.

In [ ]:
# Define a simple 2-layer neural network architecture
class ToyModel(nn.Module):
    def __init__(self, in_features=64, hidden_features=32, out_features=1):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_features, out_features)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Instantiate base model
model = ToyModel(in_features=64, hidden_features=32, out_features=1)

# Replace layer1 with our LoRA adapter module
model.layer1 = LoRALinear(model.layer1, r=4, alpha=8.0)

# Print layer hierarchy
print(model)

### Inspecting Trainable vs. Frozen Parameters

In fine-tuning scenarios, the majority of weights are frozen, leading to dramatic reductions in memory required for gradients and optimizer states (e.g., Adam's 1st and 2nd momentum vectors).

In [ ]:
print(f"{'Parameter Name':<35} | {'Shape':<20} | {'Trainable?':<10}")
print("-" * 70)

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    num_p = param.numel()
    total_params += num_p
    if param.requires_grad:
        trainable_params += num_p
    print(f"{name:<35} | {str(list(param.shape)):<20} | {str(param.requires_grad):<10}")

print("-" * 70)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Frozen Parameters:    {total_params - trainable_params:,}")

## 5. Training the LoRA Parameters

During backpropagation, only parameters where `requires_grad=True` are tracked and updated. We pass only these parameters to our optimizer.

In [ ]:
# Generate synthetic training dataset (100 samples, 64 features)
X_train = torch.randn(100, 64)
y_train = torch.randn(100, 1)

# Pass only trainable parameters to the Adam optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=0.01
)
criterion = nn.MSELoss()

# Run a 10-epoch training loop
model.train()
print("Beginning training of LoRA parameters...")
for epoch in range(1, 11):
    optimizer.zero_grad()
    preds = model(X_train)
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch:02d}/10 | Loss: {loss.item():.5f}")

## 6. Weight Merging for Production Deployment

In high-throughput serving systems, running separate adapter paths adds matrix multiplication overhead. Because matrix multiplication is distributive:

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r}(B \cdot A)$$

We can add $\Delta W$ directly into the base model weights at deployment time. The model reverts to a standard single-path neural network with **zero runtime latency overhead**.

In [ ]:
def merge_lora_to_linear(lora_layer: LoRALinear) -> nn.Linear:
    """
    Folds LoRA matrices (B @ A) * (alpha / r) directly into the base layer's weights
    and returns the original nn.Linear layer.
    """
    with torch.no_grad():
        # Compute delta W: (out_features, r) @ (r, in_features) -> (out_features, in_features)
        delta_w = (lora_layer.lora_B @ lora_layer.lora_A) * lora_layer.scaling
        
        # In-place addition into base weights
        lora_layer.base_layer.weight.data += delta_w
        
    return lora_layer.base_layer

# Test model output before merging
model.eval()
test_input = torch.randn(2, 64)
with torch.no_grad():
    out_before_merge = model(test_input)

# Merge LoRA back into standard nn.Linear
model.layer1 = merge_lora_to_linear(model.layer1)

# Test model output after merging
with torch.no_grad():
    out_after_merge = model(test_input)

# Verify numerical equivalence
max_diff = (out_before_merge - out_after_merge).abs().max().item()
print(f"Maximum numerical difference after weight merging: {max_diff:.8e}")
assert max_diff < 1e-6, "Outputs before and after merging must match identically!"
print("Weights successfully merged with zero latency impact!")
print(f"Model layer1 is now back to standard type: {type(model.layer1)}")

## 7. Production Approach: Using Hugging Face PEFT

In production environments with Transformer models (like Llama, Mistral, BERT), Hugging Face's `peft` library automates adapter injection across multi-head attention and MLP projections.

In [ ]:
try:
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForSequenceClassification
    
    # 1. Initialize pre-trained base model
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", 
        num_labels=2
    )
    
    # 2. Configure LoRA parameters
    lora_config = LoraConfig(
        r=8,                             # Rank dimension
        lora_alpha=16,                   # Scaling parameter alpha
        target_modules=["query", "value"], # Target attention projection matrices
        lora_dropout=0.05,               # Dropout regularization on adapter path
        bias="none",                     # Do not train biases
        task_type="SEQ_CLS"              # Sequence classification task
    )
    
    # 3. Wrap model with PEFT
    peft_model = get_peft_model(base_model, lora_config)
    peft_model.print_trainable_parameters()
    print("Hugging Face PEFT LoRA initialized successfully!")
except ImportError:
    print("Install peft and transformers via: pip install peft transformers")

## 8. Real LLM: LoRA on Qwen2.5-0.5B-Instruct

This section runs a real instruction model, not the toy network above.

**Scenario.** NimbusDB has an internal on-call runbook. The base model has never seen these codes. Twelve short examples teach four facts:

- `NDB-17`: replica lag over 30 seconds, fail over to `harbor-standby`
- `NDB-42`: token is read-only, mint a write-scoped token
- `NDB-9`: disk watermark at 85 percent, expand `harbor-eu-data`
- off-hours (after 22:00 UTC): page the secondary, `mira`

The base weights stay frozen. LoRA trains rank-8 adapters on the attention projections `q_proj`, `k_proj`, `v_proj`, and `o_proj`. Loss is computed only on the assistant reply, not on the question.

The questions in the before/after printout are paraphrases. They are not in the 12 training rows. A separate validation set picks the checkpoint. Two ordinary questions (`2 + 2`, capital of France) stay out of training so you can see what the adapter does to everything else.

Re-run the next cell from this folder. On this M2, with the model already cached, it takes about two minutes.


In [1]:
from pathlib import Path

script = Path("lora_real_llm_demo.py")
if not script.exists():
    raise FileNotFoundError(
        "Open or run this notebook from the project folder, next to lora_real_llm_demo.py."
    )
exec(script.read_text())


device: mps
loading Qwen/Qwen2.5-0.5B-Instruct
loaded in 3.6s

===== BEFORE LoRA (base model) =====

Q: Production just raised NDB-17. What should I do?
ACTION: Check with production support for any immediate action or instructions. MEANING: If there's no immediate response from production, consider reaching out to your team or IT department for guidance.

Q: Why would an insert return NDB-42?
An INSERT operation in NDB (Nimbus) returns a status code of 42 if there is an error during the insertion process. This typically indicates that the data was not successfully inserted into the database.

Q: The harbor volume hit NDB-9. What is the fix?
ACTION: Check NDB-9 for a known issue or maintenance. MEANING: This indicates that there may be an issue with the NDB-9 system's connectivity or performance. The fix might involve restarting the system, checking network settings, or contacting NDB-9 support for further assistance.

Q: It is 23:45 UTC and the pager went off. Who takes it?
ACTION: Ch